# PDX CyTOF — Vendi Score Analysis (cytofstandard)

Replication of `VendiScore-PDX.ipynb` using the `cytofstandard` package.

**Key differences from the original notebook:**
- No dependency on `CyTOFHelper`, `TestHet`, or other local modules
- Vendi scoring via `cytofstandard.Run.vendi_score()` with `groupby`, `markers`, `n_bins`, `n_reps`, `m`
- `n_bins` is adaptive per sample: `min(m // 2, 20)` (matches original `vendi_rarefied`)
- Results persisted to each run's zarr store

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
import scipy.linalg
import math
from pathlib import Path
from tqdm import tqdm
from sklearn.preprocessing import KBinsDiscretizer, normalize

import cytofstandard
from cytofstandard import Project

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
CLR = {'Cycling': '#3f78c1', 'Basal-like': '#fb9a99', 'Luminal': '#33a02c'}
CLASSES = ['Luminal', 'Basal-like', 'Cycling']

plt.rcParams.update({
    'axes.labelsize': 14, 'axes.titlesize': 14,
    'xtick.labelsize': 12, 'ytick.labelsize': 12,
    'figure.figsize': (6, 4), 'pdf.fonttype': 42, 'ps.fonttype': 42,
})
sns.set_style("white")

## 1. Load and preprocess raw data

In [ ]:
DATA_DIR = Path("/Users/ronguy/Dropbox/CyTOF_Breast/data/guy/with_labels/pdx/")
Numbs = sorted([5, 6, 7, 8])
DBs   = [f"PDX{N}" for N in Numbs]

Rep = dict(pd.read_excel("/Users/ronguy/Dropbox/Work/CyTOF/Mapping.xlsx").iloc[:, :].values)

raw    = {}
labels = {}

for N, DB in zip(Numbs, DBs):
    df = pd.read_parquet(DATA_DIR / f"normalized_not_scaled_{N}.0.parquet")
    df.rename(columns=Rep, inplace=True)
    df.drop(columns=[c for c in ['DNA1', 'DNA2', 'Event #'] if c in df.columns], inplace=True)

    labels[DB] = df['class'].values
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    raw[DB] = df[numeric_cols].copy()
    print(f"{DB}: {raw[DB].shape[0]} cells, {raw[DB].shape[1]} markers")

In [ ]:
# Pooled z-score — NC=1300 (≈ all cells per PDX sample)
NC = 1300
pooled = pd.concat(
    [raw[DB].sample(min(NC, len(raw[DB])), replace=False, random_state=42) for DB in DBs],
    ignore_index=True,
)
m_pool = pooled.mean()
s_pool = pooled.std().replace(0, 1)

for DB in DBs:
    raw[DB] = (raw[DB] - m_pool) / s_pool

print("Z-scoring done.")
print(pd.concat(list(raw.values())).describe().loc[['mean', 'std']].round(3))

## 2. Create cytofstandard project and ingest

In [ ]:
CYTOFSTD_DIR     = Path("/Users/ronguy/Dropbox/Work/CyTOF/Code/CyTOFSTD")
STANDARD_MARKERS = str(CYTOFSTD_DIR / "cytof_marker_registry_files/standard_markers.csv")
MARKER_ALIASES   = str(CYTOFSTD_DIR / "cytof_marker_registry_files/marker_aliases.yaml")
PROJECT_PATH     = "/Users/ronguy/Dropbox/Work/CyTOF/Projects/PDX_VendiScore"

try:
    project = Project.load(PROJECT_PATH)
    print(f"Loaded existing project at {PROJECT_PATH}")
except Exception:
    project = Project.create(
        PROJECT_PATH,
        project_id="PDX_VendiScore",
        project_name="PDX Vendi Score Analysis",
        standard_marker_file=STANDARD_MARKERS,
        marker_alias_file=MARKER_ALIASES,
    )
    print(f"Created new project at {PROJECT_PATH}")

In [ ]:
PREP_DIR = Path(PROJECT_PATH) / "preprocessed"
PREP_DIR.mkdir(parents=True, exist_ok=True)

for N, DB in zip(Numbs, DBs):
    if project.has_run(DB):
        print(f"  {DB}: already ingested — skipping")
        continue

    parquet_path = PREP_DIR / f"{DB}.parquet"
    raw[DB].to_parquet(parquet_path, index=False)

    meta_path = PREP_DIR / f"{DB}_meta.csv"
    pd.DataFrame([{
        "file_name": parquet_path.name,
        "sample_id": DB,
        "line_id":   DB,
    }]).to_csv(meta_path, index=False)

    run = project.add_run(DB, run_name=f"PDX {DB}")
    run.ingest(
        files=[str(parquet_path)],
        sample_metadata=str(meta_path),
        strict_markers=False,
        allow_extra_markers=True,
    )

    adata = run.read_adata()
    adata.obs["class"] = labels[DB]
    run._adata = adata
    run.save()
    print(f"  {DB}: ingested {adata.n_obs} cells × {adata.n_vars} markers")

project.list_runs()[['run_id', 'run_name', 'status']]

## 3. Define marker subsets

In [ ]:
ref_adata   = project.get_run(DBs[0]).read_adata()
ALL_MARKERS = ref_adata.var_names.tolist()

MRK = [m for m in ALL_MARKERS if m not in {'H3', 'H3.3', 'H4'}]

MRK_CI = [m for m in [
    'CD24', 'CD44', 'CD49f', 'E-cadherin', 'ER', 'EpCAM', 'GATA3',
    'KRT5', 'KRT8-18', 'Pan-KRT', 'Vimentin', 'aSMA',
] if m in ALL_MARKERS]

MRK_Epi = [m for m in [
    'H2AK119ub', 'H3K27ac', 'H3K27me2', 'H3K27me3',
    'H3K36me2', 'H3K36me3', 'H3K4me1', 'H3K4me3',
    'H3K64ac', 'H3K9ac', 'H3K9me2', 'H3K9me3',
    'H3S28p', 'H4K16ac', 'H4K20me3', 'pH2A.X',
] if m in ALL_MARKERS]

print(f"MRK     : {len(MRK)}")
print(f"MRK_CI  : {len(MRK_CI)}")
print(f"MRK_Epi : {len(MRK_Epi)} — {MRK_Epi}")

## 4. Vendi score per sample per class

`n_bins` is adaptive: `min(m // 2, 20)` — matches the original `vendi_rarefied` call.
PDX samples have ~1 300 cells with imbalanced classes (Luminal dominant), so `m` and `n_bins` will be small.

In [ ]:
N_REPS = 200   # raise to 5000 to match original

cache = {}

for DB in tqdm(DBs, desc="Vendi score"):
    run   = project.get_run(DB)
    adata = run.read_adata()

    # Drop zero-count categories (Categorical series gotcha)
    counts = adata.obs["class"].value_counts()
    counts = counts[counts.index.isin(CLASSES) & (counts > 0)]
    m = max(1, int(np.floor(counts.min() / 2)))
    # Adaptive n_bins: min(m//2, 20) — matches original vendi_rarefied
    n_bins = max(2, min(m // 2, 20))
    print(f"  {DB}: {dict(counts)}, m={m}, n_bins={n_bins}")

    df = run.vendi_score(
        groupby="class",
        markers=MRK_Epi,
        n_bins=n_bins,
        n_reps=N_REPS,
        m=m,
        random_state=42,
        inplace=True,
    )
    cache[DB] = df.loc[df.index.isin(CLASSES)]
    print(cache[DB].round(3))
    print()

## 5. Error-bar plot — Luminal vs Basal-like

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(DBs))

for offset, cls in [(-0.1, 'Luminal'), (0.1, 'Basal-like')]:
    means, lo, hi = [], [], []
    for DB in DBs:
        if cls not in cache[DB].index:
            means.append(np.nan); lo.append(np.nan); hi.append(np.nan)
        else:
            means.append(cache[DB].loc[cls, 'vendi_score'])
            lo.append(cache[DB].loc[cls, 'ci_low'])
            hi.append(cache[DB].loc[cls, 'ci_high'])
    means, lo, hi = map(np.array, [means, lo, hi])
    ax.errorbar(x + offset, means, yerr=[means - lo, hi - means],
                fmt='.', capsize=3, color=CLR[cls], label=cls)

ax.set_xticks(x)
ax.set_xticklabels(DBs, rotation=90)
ax.set_ylabel('Vendi Score')
ax.set_title('Vendi Score — Epigenetic Markers (PDX)')
ax.legend()
plt.tight_layout()
# plt.savefig("Plots/Vendi_EpiMRK_PDX_CyTOFSTD.pdf", dpi=200, bbox_inches='tight')
plt.show()

## 6. Grid view — all classes, all samples

In [ ]:
NCOLS = 2
nrows = math.ceil(len(DBs) / NCOLS)
fig, axes = plt.subplots(nrows, NCOLS, figsize=(5 * NCOLS, 3.5 * nrows), sharey=True)
axes = np.atleast_1d(axes).ravel()

for i, DB in enumerate(DBs):
    ax = axes[i]
    df = cache[DB]
    for cls in CLASSES:
        if cls not in df.index:
            continue
        row = df.loc[cls]
        mean, lo, hi = row['vendi_score'], row['ci_low'], row['ci_high']
        ax.errorbar([cls], [mean], yerr=[[mean - lo], [hi - mean]],
                    fmt='o', capsize=4, color=CLR[cls], label=cls)
    ax.set_title(DB)
    ax.set_ylabel('Vendi Score')
    ax.set_xticks([])

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

handles = [plt.Line2D([0], [0], marker='o', color=CLR[c], linestyle='', label=c) for c in CLASSES]
fig.legend(handles=handles, loc='lower center', ncol=len(CLASSES), frameon=False, bbox_to_anchor=(0.5, -0.04))
fig.suptitle('Vendi Score — Epigenetic Markers (PDX)', y=1.01)
plt.tight_layout()
plt.show()

## 7. Statistical test — Luminal vs Basal-like

In [ ]:
dbs_with_both = [DB for DB in DBs
                 if 'Luminal' in cache[DB].index and 'Basal-like' in cache[DB].index]

means_L  = np.array([cache[DB].loc['Luminal',    'vendi_score'] for DB in dbs_with_both])
means_BL = np.array([cache[DB].loc['Basal-like', 'vendi_score'] for DB in dbs_with_both])

stat, pval = scipy.stats.mannwhitneyu(means_L, means_BL, alternative='two-sided')
print(f"Samples with both classes: {dbs_with_both}")
print(f"Mann-Whitney U:  U = {stat:.1f},  p = {pval:.4f}")
print(f"Luminal     mean ± std:  {means_L.mean():.3f} ± {means_L.std():.3f}")
print(f"Basal-like  mean ± std:  {means_BL.mean():.3f} ± {means_BL.std():.3f}")

## 8. Cell Identity markers (MRK_CI)

In [ ]:
cache_CI = {}

for DB in tqdm(DBs, desc="Vendi (CI markers)"):
    run   = project.get_run(DB)
    adata = run.read_adata()
    counts = adata.obs["class"].value_counts()
    counts = counts[counts.index.isin(CLASSES) & (counts > 0)]
    m = max(1, int(np.floor(counts.min() / 2)))
    n_bins = max(2, min(m // 2, 20))

    df = run.vendi_score(
        groupby="class",
        markers=MRK_CI,
        obs_key="vendi_score_CI",
        n_bins=n_bins,
        n_reps=N_REPS,
        m=m,
        random_state=42,
        inplace=True,
    )
    cache_CI[DB] = df.loc[df.index.isin(CLASSES)]

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(DBs))
for offset, cls in [(-0.1, 'Luminal'), (0.1, 'Basal-like')]:
    means, lo, hi = [], [], []
    for DB in DBs:
        if cls not in cache_CI[DB].index:
            means.append(np.nan); lo.append(np.nan); hi.append(np.nan)
        else:
            means.append(cache_CI[DB].loc[cls, 'vendi_score'])
            lo.append(cache_CI[DB].loc[cls, 'ci_low'])
            hi.append(cache_CI[DB].loc[cls, 'ci_high'])
    means, lo, hi = map(np.array, [means, lo, hi])
    ax.errorbar(x + offset, means, yerr=[means - lo, hi - means],
                fmt='.', capsize=3, color=CLR[cls], label=cls)
ax.set_xticks(x); ax.set_xticklabels(DBs, rotation=90)
ax.set_ylabel('Vendi Score'); ax.legend()
ax.set_title('Vendi Score — Cell Identity Markers (PDX)')
plt.tight_layout(); plt.show()

## 9. Eigenvalue spectrum per class

In [ ]:
evals_per_sample = {}

for DB in tqdm(DBs, desc="Eigenvalues"):
    run   = project.get_run(DB)
    adata = run.read_adata()
    counts = adata.obs["class"].value_counts()
    counts = counts[counts.index.isin(CLASSES) & (counts > 0)]
    m = max(1, int(np.floor(counts.min() / 2)))
    n_bins = max(2, min(m // 2, 20))

    _, evals = run.vendi_score(
        groupby="class",
        markers=MRK_Epi,
        obs_key="vendi_score",
        n_bins=n_bins,
        n_reps=1,
        m=m,
        return_eigenvalues=True,
        inplace=False,
    )
    evals_per_sample[DB] = {cls: evals[cls] for cls in CLASSES if cls in evals}

# Overlay plot
fig, ax = plt.subplots(figsize=(7, 4))
for cls in CLASSES:
    available = [evals_per_sample[DB][cls] for DB in DBs if cls in evals_per_sample[DB]]
    if not available:
        continue
    stacked = np.vstack(available)
    stacked_sorted = np.sort(stacked, axis=1)[:, ::-1]
    mean_evals = stacked_sorted.mean(axis=0)
    ax.plot(np.arange(len(mean_evals)), mean_evals, 'o-', color=CLR[cls], label=cls, markersize=4)

ax.set_xlabel('Eigenvalue rank')
ax.set_ylabel('Mean eigenvalue')
ax.set_title('Eigenvalue spectrum — Epigenetic Markers (PDX)')
ax.legend()
plt.yscale('log')
plt.tight_layout()
plt.show()

## 10. Per-cell Vendi score (k = 15)

In [ ]:
K = 15
OBS_KEY = "vendi_percell"
obs_all = []

for DB in tqdm(DBs, desc="Per-cell Vendi"):
    run = project.get_run(DB)
    run.vendi_score(
        k=K,
        markers=MRK_Epi,
        n_bins=10,
        obs_key=OBS_KEY,
        metric="cosine",
        inplace=True,
    )
    adata = run.read_adata()
    df = adata.obs[["class", OBS_KEY]].copy()
    df["sample"] = DB
    obs_all.append(df)

obs_all = pd.concat(obs_all, ignore_index=True)
print(f"Total cells: {len(obs_all)}")

In [ ]:
summary = (
    obs_all[obs_all["class"].isin(CLASSES)]
    .groupby("class")[OBS_KEY]
    .agg(mean="mean", std="std", n="count")
    .reindex([c for c in CLASSES if c in obs_all["class"].unique()])
    .round(3)
)
print(f"Per-cell Vendi (k={K}) — mean ± std per class (pooled across PDX samples)")
print(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Violin
plot_df = obs_all[obs_all["class"].isin(CLASSES)]
present = [c for c in CLASSES if c in plot_df["class"].unique()]
sns.violinplot(data=plot_df, x="class", y=OBS_KEY, order=present,
               palette=CLR, inner="box", linewidth=0.8, ax=axes[0])
axes[0].set_xlabel("")
axes[0].set_ylabel(f"Per-cell Vendi (k={K})")
axes[0].set_title("Distribution per class (PDX)")

# Per-sample means
per_sample = (
    obs_all[obs_all["class"].isin(CLASSES)]
    .groupby(["sample", "class"])[OBS_KEY]
    .mean()
    .unstack("class")
    .reindex(DBs)
)
x = np.arange(len(DBs))
for offset, cls in [(-0.1, "Luminal"), (0.1, "Basal-like"), (0.0, "Cycling")]:
    if cls not in per_sample.columns:
        continue
    axes[1].plot(x + offset, per_sample[cls].values, "o", color=CLR[cls], label=cls, markersize=6)
axes[1].set_xticks(x); axes[1].set_xticklabels(DBs, rotation=90)
axes[1].set_ylabel(f"Mean per-cell Vendi (k={K})")
axes[1].set_title("Mean per sample (PDX)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 11. Marker ablation — leave-one-out Vendi score

For each epigenetic marker, recompute Vendi with that marker removed.
**delta = baseline − LOO**: positive → marker drives diversity; negative → marker was compressing the score.

In [ ]:
N_REPS_LOO = 50

loo_scores = {}

for dropped in tqdm(MRK_Epi, desc="LOO"):
    markers_loo = [mk for mk in MRK_Epi if mk != dropped]
    loo_scores[dropped] = {}

    for DB in DBs:
        run   = project.get_run(DB)
        adata = run.read_adata()
        counts = adata.obs["class"].value_counts()
        counts = counts[counts.index.isin(CLASSES) & (counts > 0)]
        m = max(1, int(np.floor(counts.min() / 2)))
        n_bins = max(2, min(m // 2, 20))

        df = run.vendi_score(
            groupby="class",
            markers=markers_loo,
            n_bins=n_bins,
            n_reps=N_REPS_LOO,
            m=m,
            random_state=42,
            inplace=False,
        )
        loo_scores[dropped][DB] = (
            df.loc[df.index.isin(CLASSES), "vendi_score"].to_dict()
        )

# Build delta DataFrame
records = []
for dropped in MRK_Epi:
    for cls in CLASSES:
        deltas = [
            cache[DB].loc[cls, "vendi_score"] - loo_scores[dropped][DB][cls]
            for DB in DBs
            if cls in cache[DB].index and cls in loo_scores[dropped].get(DB, {})
        ]
        if deltas:
            records.append({"marker": dropped, "class": cls,
                            "delta_mean": np.mean(deltas), "delta_std": np.std(deltas)})

delta_df = (
    pd.DataFrame(records)
    .pivot(index="marker", columns="class", values="delta_mean")
    .reindex(index=MRK_Epi, columns=CLASSES)
)
delta_df = delta_df.loc[delta_df.abs().sum(axis=1).sort_values(ascending=False).index]

# Heatmap
fig, ax = plt.subplots(figsize=(4, 7))
vmax = delta_df.abs().max().max()
sns.heatmap(delta_df, ax=ax, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
            annot=True, fmt=".2f", annot_kws={"size": 9}, linewidths=0.4,
            cbar_kws={"label": "Δ Vendi (baseline − LOO)", "shrink": 0.6})
ax.set_title("Marker ablation — Epigenetic Vendi (PDX)\n(+: drives diversity)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

# Bar chart per class
fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 5), sharey=True)
for ax, cls in zip(axes, CLASSES):
    vals = delta_df[cls].dropna().sort_values()
    ax.barh(vals.index, vals.values,
            color=["#d73027" if v > 0 else "#4575b4" for v in vals])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(cls)
    ax.set_xlabel("Δ Vendi")
    if ax is axes[0]:
        ax.set_ylabel("Marker")
fig.suptitle("Per-class marker ablation (PDX)", y=1.01)
plt.tight_layout()
plt.show()

## 12. Marker contributions to individual eigenvectors

Squared loadings `v_j[k]²` show which markers dominate each diversity direction.
Entropy-weighted: `(−λ_j log λ_j) × v_j[k]²` is each (marker, eigenvector) pair's actual contribution to the Vendi score.

In [ ]:
import warnings as _warnings

eig_data = {}

for cls in CLASSES:
    eig_data[cls] = {}
    for DB in DBs:
        run   = project.get_run(DB)
        adata = run.read_adata()

        var_names = adata.var_names.tolist()
        col_idx   = [var_names.index(mk) for mk in MRK_Epi if mk in var_names]
        markers_present = [mk for mk in MRK_Epi if mk in var_names]

        X    = np.asarray(adata.X, dtype=np.float64)
        pool = X[(adata.obs["class"] == cls).values][:, col_idx]

        if len(pool) < 2:
            continue

        counts = adata.obs["class"].value_counts()
        counts = counts[counts.index.isin(CLASSES) & (counts > 0)]
        m = max(1, int(np.floor(counts.min() / 2)))
        n_bins = max(2, min(m // 2, 20))

        rng = np.random.default_rng(42)
        sub = pool[rng.choice(len(pool), min(m, len(pool)), replace=False)]

        with _warnings.catch_warnings():
            _warnings.filterwarnings("ignore")
            binner = KBinsDiscretizer(n_bins=n_bins, strategy="uniform", encode="ordinal")
            MM = binner.fit_transform(sub)

        MN = normalize(MM, axis=1)
        S  = MN.T @ MN / len(MN)
        w, V = scipy.linalg.eigh(S)
        w = w[::-1]; V = V[:, ::-1]

        eig_data[cls][DB] = {"w": w, "V2": V**2, "markers": markers_present}

# Aggregate across samples
agg = {}
for cls in CLASSES:
    available = [DB for DB in DBs if DB in eig_data[cls]]
    if not available:
        continue
    agg[cls] = {
        "mean_V2": np.mean([eig_data[cls][DB]["V2"] for DB in available], axis=0),
        "mean_w":  np.mean([eig_data[cls][DB]["w"]  for DB in available], axis=0),
    }

N_SHOW = min(8, len(MRK_Epi))

# View 1: squared loadings
fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 7), sharey=True)
for ax, cls in zip(axes, CLASSES):
    if cls not in agg:
        ax.axis("off"); continue
    V2 = agg[cls]["mean_V2"][:, :N_SHOW]
    w  = agg[cls]["mean_w"][:N_SHOW]
    df = pd.DataFrame(V2, index=MRK_Epi,
                      columns=[f"EV{i+1}  λ={w[i]:.3f}" for i in range(N_SHOW)])
    sns.heatmap(df, ax=ax, cmap="YlOrRd", vmin=0, vmax=1,
                annot=True, fmt=".2f", annot_kws={"size": 7}, linewidths=0.3,
                cbar_kws={"label": "v²ⱼ[k]", "shrink": 0.5})
    ax.set_title(cls); ax.set_ylabel("Marker" if ax is axes[0] else "")
    ax.tick_params(axis="x", rotation=45)
fig.suptitle("Marker squared loadings on eigenvectors (PDX)", y=1.01)
plt.tight_layout(); plt.show()

# View 2: entropy-weighted loadings
ew_dfs = {}
vmax_all = 0.0
for cls in CLASSES:
    if cls not in agg:
        continue
    w  = agg[cls]["mean_w"][:N_SHOW]
    V2 = agg[cls]["mean_V2"][:, :N_SHOW]
    with np.errstate(divide="ignore", invalid="ignore"):
        hw = np.where(w > 0, -w * np.log(w), 0.0)
    ew = V2 * hw[np.newaxis, :]
    ew_dfs[cls] = pd.DataFrame(ew, index=MRK_Epi,
                               columns=[f"EV{i+1}  λ={w[i]:.3f}" for i in range(N_SHOW)])
    vmax_all = max(vmax_all, ew.max())

fig, axes = plt.subplots(1, len(CLASSES), figsize=(5 * len(CLASSES), 7), sharey=True)
for ax, cls in zip(axes, CLASSES):
    if cls not in ew_dfs:
        ax.axis("off"); continue
    sns.heatmap(ew_dfs[cls], ax=ax, cmap="PuRd", vmin=0, vmax=vmax_all,
                annot=True, fmt=".3f", annot_kws={"size": 7}, linewidths=0.3,
                cbar_kws={"label": "(−λⱼ log λⱼ)·v²ⱼ[k]", "shrink": 0.5})
    ax.set_title(cls); ax.set_ylabel("Marker" if ax is axes[0] else "")
    ax.tick_params(axis="x", rotation=45)
fig.suptitle("Entropy-weighted marker loadings (PDX)", y=1.01)
plt.tight_layout(); plt.show()

# View 3: total marker contribution
total_contrib = {cls: ew_dfs[cls].values.sum(axis=1) for cls in CLASSES if cls in ew_dfs}
total_df = pd.DataFrame(total_contrib, index=MRK_Epi)
total_df = total_df.reindex(columns=[c for c in CLASSES if c in total_df.columns])
total_df = total_df.loc[total_df.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(4, 6))
sns.heatmap(total_df, ax=ax, cmap="YlOrRd", annot=True, fmt=".3f",
            annot_kws={"size": 9}, linewidths=0.4,
            cbar_kws={"label": "Total entropy contribution", "shrink": 0.6})
ax.set_title("Total marker → Vendi entropy\ncontribution (PDX)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## 13. λ₁ vs Vendi score — Basal-like cells, per PDX sample

Same analysis as the BRCA notebook: each point is one PDX sample.
λ₁ uses all Basal-like cells in the sample (no subsampling).

In [ ]:
TARGET_CLS = "Basal-like"
import warnings as _w2

lambda1_vals = {}
vs_vals      = {}

for DB in DBs:
    if TARGET_CLS not in cache[DB].index:
        continue

    run   = project.get_run(DB)
    adata = run.read_adata()
    var_names = adata.var_names.tolist()
    col_idx   = [var_names.index(mk) for mk in MRK_Epi if mk in var_names]

    X    = np.asarray(adata.X, dtype=np.float64)
    pool = X[(adata.obs["class"] == TARGET_CLS).values][:, col_idx]

    if len(pool) < 2:
        continue

    counts = adata.obs["class"].value_counts()
    counts = counts[counts.index.isin(CLASSES) & (counts > 0)]
    m = max(1, int(np.floor(counts.min() / 2)))
    n_bins = max(2, min(m // 2, 20))

    with _w2.catch_warnings():
        _w2.filterwarnings("ignore")
        binner = KBinsDiscretizer(n_bins=n_bins, strategy="uniform", encode="ordinal")
        MM = binner.fit_transform(pool)

    MN = normalize(MM, axis=1)
    S  = MN.T @ MN / len(MN)
    w  = scipy.linalg.eigvalsh(S)

    lambda1_vals[DB] = float(w[-1])
    vs_vals[DB]      = cache[DB].loc[TARGET_CLS, "vendi_score"]

fig, ax = plt.subplots(figsize=(5, 5))

dbs_plot = [DB for DB in DBs if DB in lambda1_vals]
x_vals   = np.array([lambda1_vals[DB] for DB in dbs_plot])
y_vals   = np.array([vs_vals[DB]      for DB in dbs_plot])

ax.scatter(x_vals, y_vals, color=CLR[TARGET_CLS], s=100, zorder=3)
for DB, x, y in zip(dbs_plot, x_vals, y_vals):
    ax.annotate(DB, (x, y), textcoords="offset points", xytext=(5, 3), fontsize=10)

if len(x_vals) > 2:
    r, p = scipy.stats.pearsonr(x_vals, y_vals)
    ax.text(0.05, 0.95, f"r = {r:.2f},  p = {p:.3f}",
            transform=ax.transAxes, va="top", fontsize=10)

ax.set_xlabel("λ₁  (largest dual-kernel eigenvalue)")
ax.set_ylabel("Vendi Score")
ax.set_title(f"λ₁ vs Vendi Score — {TARGET_CLS}\n(PDX samples, all Basal-like cells)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(pd.DataFrame({"λ₁": lambda1_vals, "VS": vs_vals}).round(4))